# 01 · Exploración del dataset WOS-46985 (Fase I)

Revisión de estructura, categorías, distribución de clases, longitud de los resúmenes y datos faltantes.
Las decisiones de limpieza que salen de aquí están en `docs/decisiones/0002-etiquetas-wos.md` y se implementan en `clasificador_ml.datos`.

Requisito: el dataset descargado en `data/raw/` (ver `data/README.md`).

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from clasificador_ml.datos import construir_etiquetas, leer_wos, limpiar
from clasificador_ml.rutas import DATOS_PROCESADOS

crudo = leer_wos()
crudo.head()

## Estructura y datos faltantes

In [ ]:
print(crudo.shape)
crudo.isna().sum()

In [ ]:
print('Resúmenes duplicados:', crudo['Abstract'].str.strip().str.lower().duplicated().sum())

## Inconsistencias de etiquetas

¿Cada `Y` (área global) pertenece a un solo dominio? ¿Tiene un solo nombre?

In [ ]:
df = limpiar(crudo)
porY = df.groupby('Y').agg(dominios=('Domain', 'unique'), nombres=('area', 'unique'), n=('Y', 'size'))
porY[(porY['dominios'].str.len() > 1) | (porY['nombres'].str.len() > 1)]

## Distribución de clases

In [ ]:
etiquetado, categorias = construir_etiquetas(df)
print(categorias['total_dominios'], 'dominios ·', categorias['total_areas'], 'áreas')

fig, ax = plt.subplots(figsize=(8, 4))
etiquetado['dominio'].value_counts().plot.bar(ax=ax)
ax.set_title('Documentos por dominio')
ax.set_ylabel('Documentos')
plt.tight_layout()

In [ ]:
por_area = etiquetado['id_area'].value_counts()
print(por_area.describe())

fig, ax = plt.subplots(figsize=(10, 4))
por_area.sort_values(ascending=False).reset_index(drop=True).plot(ax=ax)
ax.set_title('Documentos por área (ordenadas)')
ax.set_xlabel('Área')
ax.set_ylabel('Documentos')
plt.tight_layout()

In [ ]:
etiquetado.groupby('dominio')['id_area'].nunique().sort_values()

## Longitud de los resúmenes

In [ ]:
palabras = etiquetado['texto'].str.split().str.len()
print(palabras.describe())

fig, ax = plt.subplots(figsize=(8, 4))
palabras.clip(upper=600).hist(bins=60, ax=ax)
ax.set_title('Palabras por resumen (recortado a 600)')
plt.tight_layout()

In [ ]:
etiquetado.loc[palabras < 20, ['dominio', 'area', 'texto']]

## Particiones generadas

Tras ejecutar `python -m clasificador_ml.datos`:

In [ ]:
categorias_guardadas = json.loads((DATOS_PROCESADOS / 'categorias.json').read_text(encoding='utf-8'))
{k: v for k, v in categorias_guardadas.items() if k != 'dominios'}